# 🌱 CSIRO Image2Biomass - V3 (Data Structure Fixed)

**Fixes:**
- ✅ Uses `sample_id` instead of `id`
- ✅ Handles actual competition data structure
- ✅ Proper error handling for missing images

⚙️ **IMPORTANT Notebook Settings**:
- ✅ Accelerator: **GPU T4 x2** (MUST enable!)
- ✅ Internet: **ON**
- Persistence: Files only

In [ ]:
%%time
# Install only timm
!pip install -q timm==0.9.12

import torch
print(f"✅ Setup complete!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU NOT AVAILABLE! Training will be VERY slow.")
    print("Please enable GPU: Settings → Accelerator → GPU T4 x2")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

class BiomassDataset(Dataset):
    def __init__(self, df, image_dir, img_size=384, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.img_size = img_size
        self.transform = transform
        self.is_train = is_train
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Handle different column names
        img_id = row.get('sample_id', row.get('id', idx))
        
        # Get image path
        if 'image_path' in row.index:
            img_name = row['image_path']
        else:
            img_name = f"{img_id}.jpg"
        
        img_path = os.path.join(self.image_dir, img_name)
        
        # Load image with error handling
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Create dummy image if file not found
            image = Image.new('RGB', (self.img_size, self.img_size), (128, 128, 128))
        
        if self.transform:
            image = self.transform(image)
        
        sample = {'image': image, 'id': img_id}
        
        if self.is_train and 'target' in row.index:
            sample['target'] = torch.tensor([row['target']], dtype=torch.float32)
        
        return sample

def get_train_transforms(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_valid_transforms(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

print("✅ Dataset defined")

In [ ]:
import timm

class BiomassModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, 
                                         num_classes=0, global_pool='avg')
        
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            features = self.backbone(dummy)
            feat_dim = features.shape[1]
        
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        if isinstance(x, dict):
            x = x['image']
        features = self.backbone(x)
        return self.head(features)

print("✅ Model defined")

In [ ]:
# Configuration
CONFIG = {
    'data_dir': '/kaggle/input/csiro-biomass',
    'img_size': 384,
    'batch_size': 16,
    'num_epochs': 15,
    'lr': 3e-4,
    'n_folds': 5,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"Device: {CONFIG['device']}\n")

# Load and inspect data
train_df = pd.read_csv(f"{CONFIG['data_dir']}/train.csv")
print(f"Train samples: {len(train_df)}")
print(f"Columns: {list(train_df.columns)}")
print("\nData structure:")
print(train_df.head())
print("\nTarget statistics:")
print(train_df['target'].describe())

In [ ]:
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    losses = AverageMeter()
    
    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        images = batch['image'].to(device)
        targets = batch['target'].to(device)
        
        with torch.amp.autocast(device_type=device):
            preds = model(images)
            loss = criterion(preds, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        losses.update(loss.item(), images.size(0))
        pbar.set_postfix({'loss': f'{losses.avg:.4f}'})
    
    return losses.avg

def validate(model, loader, criterion, device):
    model.eval()
    losses = AverageMeter()
    preds_list = []
    targets_list = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            images = batch['image'].to(device)
            targets = batch['target'].to(device)
            
            preds = model(images)
            loss = criterion(preds, targets)
            
            losses.update(loss.item(), images.size(0))
            preds_list.append(preds.cpu().numpy())
            targets_list.append(targets.cpu().numpy())
    
    preds_list = np.concatenate(preds_list)
    targets_list = np.concatenate(targets_list)
    rmse = np.sqrt(np.mean((preds_list - targets_list)**2))
    
    return losses.avg, rmse

def simple_kfold(df, n_splits=5, seed=42):
    """Manual K-Fold without sklearn"""
    np.random.seed(seed)
    indices = np.arange(len(df))
    np.random.shuffle(indices)
    
    fold_size = len(df) // n_splits
    
    for fold in range(n_splits):
        val_start = fold * fold_size
        val_end = (fold + 1) * fold_size if fold < n_splits - 1 else len(df)
        
        val_idx = indices[val_start:val_end]
        train_idx = np.concatenate([indices[:val_start], indices[val_end:]])
        
        yield fold, train_idx, val_idx

print("✅ Training functions defined")

In [ ]:
%%time
# Create checkpoint directory
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

fold_scores = []

for fold, train_idx, valid_idx in simple_kfold(train_df, CONFIG['n_folds'], CONFIG['seed']):
    print(f"\n{'='*60}")
    print(f"Fold {fold}")
    print(f"{'='*60}\n")
    
    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    valid_data = train_df.iloc[valid_idx].reset_index(drop=True)
    
    train_dataset = BiomassDataset(
        train_data, f"{CONFIG['data_dir']}/",
        CONFIG['img_size'], get_train_transforms(CONFIG['img_size']), is_train=True
    )
    
    valid_dataset = BiomassDataset(
        valid_data, f"{CONFIG['data_dir']}/",
        CONFIG['img_size'], get_valid_transforms(CONFIG['img_size']), is_train=True
    )
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                            shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=CONFIG['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)
    
    model = BiomassModel('efficientnet_b3', pretrained=True, dropout=0.3)
    model = model.to(CONFIG['device'])
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])
    criterion = nn.SmoothL1Loss()
    scaler = torch.amp.GradScaler(CONFIG['device'])
    
    best_rmse = float('inf')
    patience = 7
    patience_counter = 0
    
    for epoch in range(CONFIG['num_epochs']):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, 
                                    CONFIG['device'], scaler)
        valid_loss, valid_rmse = validate(model, valid_loader, criterion, CONFIG['device'])
        
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, Valid RMSE: {valid_rmse:.4f}")
        
        if valid_rmse < best_rmse:
            best_rmse = valid_rmse
            patience_counter = 0
            torch.save({
                'model': model.state_dict(),
                'rmse': valid_rmse,
                'epoch': epoch
            }, f'/kaggle/working/checkpoints/best_fold{fold}.pth')
            print(f"✅ Best model saved! RMSE: {valid_rmse:.4f}")
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    fold_scores.append(best_rmse)
    print(f"\nFold {fold} Best RMSE: {best_rmse:.4f}")

print(f"\n{'='*60}")
print(f"Mean CV RMSE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"{'='*60}")

## ⚠️ Turn OFF Internet Now

Before running inference:
1. Settings → Internet → **OFF**
2. Save settings
3. Continue below

In [ ]:
# Load test data
test_df = pd.read_csv(f"{CONFIG['data_dir']}/test.csv")
print(f"Test samples: {len(test_df)}")
print(f"Test columns: {list(test_df.columns)}")

test_dataset = BiomassDataset(
    test_df, f"{CONFIG['data_dir']}/",
    CONFIG['img_size'], get_valid_transforms(CONFIG['img_size']), is_train=False
)

test_loader = DataLoader(test_dataset, batch_size=32, 
                        shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
%%time
all_predictions = []

for fold in range(CONFIG['n_folds']):
    checkpoint_path = f"/kaggle/working/checkpoints/best_fold{fold}.pth"
    
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint for fold {fold} not found, skipping")
        continue
    
    print(f"Loading fold {fold}...")
    model = BiomassModel('efficientnet_b3', pretrained=False)
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model'])
    model = model.to(CONFIG['device'])
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Fold {fold}"):
            images = batch['image'].to(CONFIG['device'])
            preds = model(images)
            fold_preds.append(preds.cpu().numpy())
    
    fold_preds = np.concatenate(fold_preds)
    all_predictions.append(fold_preds)
    print(f"Fold {fold} RMSE: {checkpoint['rmse']:.4f}")

if len(all_predictions) > 0:
    final_predictions = np.mean(all_predictions, axis=0).squeeze()
    print(f"\n✅ Generated {len(final_predictions)} predictions")
else:
    print("\n❌ No predictions generated - no trained models found!")

In [ ]:
# Create submission - FIXED for actual data structure
id_col = 'sample_id' if 'sample_id' in test_df.columns else 'id'
target_col = 'target' if 'target' in test_df.columns else 'biomass'

submission = pd.DataFrame({
    id_col: test_df[id_col],
    target_col: final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print("✅ Submission created!\n")
print("Preview:")
print(submission.head(10))
print(f"\nStatistics:")
print(submission[target_col].describe())
print(f"\n💾 Saved to: /kaggle/working/submission.csv")
print("📥 Download from Output tab!")